In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
import os

os.chdir(r"D:\business_analytics_project1")

print(os.getcwd())

D:\business_analytics_project1


In [3]:
parts_master = pd.read_csv("data/raw/parts_master.csv")
purchase_orders = pd.read_csv("data/raw/purchase_orders.csv")
quality_incidents = pd.read_csv("data/raw/quality_incidents.csv")
supply_chain_history = pd.read_csv("data/raw/supply_chain_history.csv")

In [4]:
datasets = {
    "Parts Master": parts_master,
    "Purchase Orders": purchase_orders,
    "Quality Incidents": quality_incidents,
    "Supply Chain History": supply_chain_history
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Parts Master: (300, 9)
Purchase Orders: (29666, 9)
Quality Incidents: (368, 8)
Supply Chain History: (280800, 11)


In [5]:
# Convert date columns to datetime

purchase_orders["order_date"] = pd.to_datetime(purchase_orders["order_date"])
purchase_orders["promised_date"] = pd.to_datetime(purchase_orders["promised_date"])
purchase_orders["receipt_date"] = pd.to_datetime(purchase_orders["receipt_date"])

quality_incidents["incident_date"] = pd.to_datetime(
    quality_incidents["incident_date"]
)

supply_chain_history["date"] = pd.to_datetime(
    supply_chain_history["date"]
)

In [6]:
print(purchase_orders[[
    "order_date",
    "promised_date",
    "receipt_date"
]].dtypes)

print("\nQuality Incidents:")
print(quality_incidents["incident_date"].dtype)

print("\nSupply Chain History:")
print(supply_chain_history["date"].dtype)

order_date       datetime64[us]
promised_date    datetime64[us]
receipt_date     datetime64[us]
dtype: object

Quality Incidents:
datetime64[us]

Supply Chain History:
datetime64[us]


In [7]:
shelf_life_analysis = parts_master.copy()

shelf_life_analysis["shelf_life_missing"] = (
    shelf_life_analysis["shelf_life_days"].isna()
)

print("Missing shelf life by part family:")
display(
    pd.crosstab(
        shelf_life_analysis["part_family"],
        shelf_life_analysis["shelf_life_missing"]
    )
)

print("\nMissing shelf life by criticality:")
display(
    pd.crosstab(
        shelf_life_analysis["criticality_class"],
        shelf_life_analysis["shelf_life_missing"]
    )
)

print("\nMissing shelf life by repairability:")
display(
    pd.crosstab(
        shelf_life_analysis["is_repairable"],
        shelf_life_analysis["shelf_life_missing"]
    )
)

Missing shelf life by part family:


shelf_life_missing,False,True
part_family,,
Avionics,0,37
Cabin,0,33
Electrical,15,26
Engine,0,32
Fasteners,0,61
Hydraulics,11,22
LandingGear,0,27
Structure,0,36



Missing shelf life by criticality:


shelf_life_missing,False,True
criticality_class,,
A,4,44
B,10,85
C,12,145



Missing shelf life by repairability:


shelf_life_missing,False,True
is_repairable,,
No,16,200
Yes,10,74


# shelf life treatment 
shelf_life_days contains 274 missing values out of 300 parts (91.33%). The missingness is concentrated across specific part families and is therefore not suitable for simple statistical imputation.

The original values will be retained, while missing values will be treated as unknown. The variable will not be used as a primary supplier-risk feature.


In [8]:
purchase_orders["delivery_delay_days"] = (
    purchase_orders["receipt_date"] -
    purchase_orders["promised_date"]
).dt.days

purchase_orders[[
    "po_id",
    "supplier_id",
    "part_id",
    "promised_date",
    "receipt_date",
    "delivery_delay_days"
]].head(10)

,po_id,supplier_id,part_id,promised_date,receipt_date,delivery_delay_days
0,PO000001,SUP004,P00001,2022-05-28,2022-05-29,1
1,PO000002,SUP004,P00001,2022-07-31,2022-08-02,2
2,PO000003,SUP004,P00001,2022-11-12,2022-11-16,4
3,PO000004,SUP004,P00001,2023-03-07,2023-03-09,2
4,PO000005,SUP004,P00001,2023-04-28,2023-04-29,1
5,PO000006,SUP004,P00001,2023-12-17,2023-12-17,0
6,PO000007,SUP004,P00001,2023-12-31,2024-01-01,1
7,PO000008,SUP004,P00001,2024-05-26,2024-05-30,4
8,PO000009,SUP004,P00001,2024-06-22,2024-06-21,-1
9,PO000010,SUP004,P00001,2024-08-21,2024-08-21,0


In [9]:
print("Early deliveries:", 
      (purchase_orders["delivery_delay_days"] < 0).sum())

print("On-time deliveries:",
      (purchase_orders["delivery_delay_days"] == 0).sum())

print("Late deliveries:",
      (purchase_orders["delivery_delay_days"] > 0).sum())

Early deliveries: 9555
On-time deliveries: 3543
Late deliveries: 16568


In [10]:
purchase_orders["late_delivery"] = (
    purchase_orders["delivery_delay_days"] > 0
).astype(int)

purchase_orders[[
    "po_id",
    "supplier_id",
    "delivery_delay_days",
    "late_delivery"
]].head(10)

,po_id,supplier_id,delivery_delay_days,late_delivery
0,PO000001,SUP004,1,1
1,PO000002,SUP004,2,1
2,PO000003,SUP004,4,1
3,PO000004,SUP004,2,1
4,PO000005,SUP004,1,1
5,PO000006,SUP004,0,0
6,PO000007,SUP004,1,1
7,PO000008,SUP004,4,1
8,PO000009,SUP004,-1,0
9,PO000010,SUP004,0,0


In [11]:
print(purchase_orders["late_delivery"].value_counts())

late_delivery
1    16568
0    13098
Name: count, dtype: int64


In [12]:
supplier_delivery = (
    purchase_orders
    .groupby("supplier_id")
    .agg(
        total_orders=("po_id", "count"),
        late_orders=("late_delivery", "sum"),
        late_delivery_rate=("late_delivery", "mean"),
        avg_delivery_delay=("delivery_delay_days", "mean"),
        max_delivery_delay=("delivery_delay_days", "max")
    )
    .reset_index()
)

supplier_delivery["late_delivery_rate"] = (
    supplier_delivery["late_delivery_rate"] * 100
).round(2)

supplier_delivery["avg_delivery_delay"] = (
    supplier_delivery["avg_delivery_delay"].round(2)
)

supplier_delivery
supplier_delivery.sort_values(
    "late_delivery_rate",
    ascending=False
).head(10)

,supplier_id,total_orders,late_orders,late_delivery_rate,avg_delivery_delay,max_delivery_delay
32,SUP033,631,608,96.35,6.09,17
22,SUP023,280,206,73.57,2.37,11
1,SUP002,763,548,71.82,2.25,12
25,SUP026,680,480,70.59,2.16,14
38,SUP039,306,215,70.26,2.18,12
14,SUP015,413,290,70.22,2.21,11
20,SUP021,523,367,70.17,2.08,10
8,SUP009,649,453,69.80,1.93,12
13,SUP014,843,583,69.16,2.08,13
24,SUP025,543,375,69.06,2.07,11


In [13]:
purchase_orders["actual_lead_time_days"] = (
    purchase_orders["receipt_date"] -
    purchase_orders["order_date"]
).dt.days
print(purchase_orders["actual_lead_time_days"].describe())
print(
    "Orders with negative actual lead time:",
    (purchase_orders["actual_lead_time_days"] < 0).sum()
)

count    29666.000000
mean        42.971516
std         15.862033
min          8.000000
25%         32.000000
50%         41.000000
75%         52.000000
max        131.000000
Name: actual_lead_time_days, dtype: float64
Orders with negative actual lead time: 0


In [18]:
purchase_orders = purchase_orders.merge(
    parts_master[["part_id", "lead_time_days"]],
    on="part_id",
    how="left"
)

In [19]:
purchase_orders["lead_time_variance_days"] = (
    purchase_orders["actual_lead_time_days"]
    - purchase_orders["lead_time_days"]
)

In [20]:
purchase_orders[[
    "part_id",
    "lead_time_days",
    "actual_lead_time_days",
    "lead_time_variance_days"
]].head(10)

,part_id,lead_time_days,actual_lead_time_days,lead_time_variance_days
0,P00001,27,27,0
1,P00001,27,29,2
2,P00001,27,30,3
3,P00001,27,31,4
4,P00001,27,26,-1
5,P00001,27,27,0
6,P00001,27,28,1
7,P00001,27,31,4
8,P00001,27,25,-2
9,P00001,27,30,3


In [21]:
supplier_lead_time = (
    purchase_orders
    .groupby("supplier_id")
    .agg(
        avg_actual_lead_time=("actual_lead_time_days", "mean"),
        avg_expected_lead_time=("lead_time_days", "mean"),
        avg_lead_time_variance=("lead_time_variance_days", "mean"),
        lead_time_std=("actual_lead_time_days", "std")
    )
    .reset_index()
)

supplier_lead_time["avg_actual_lead_time"] = (
    supplier_lead_time["avg_actual_lead_time"].round(2)
)

supplier_lead_time["avg_expected_lead_time"] = (
    supplier_lead_time["avg_expected_lead_time"].round(2)
)

supplier_lead_time["avg_lead_time_variance"] = (
    supplier_lead_time["avg_lead_time_variance"].round(2)
)

supplier_lead_time["lead_time_std"] = (
    supplier_lead_time["lead_time_std"].round(2)
)

supplier_lead_time.sort_values(
    "avg_lead_time_variance",
    ascending=False
).head(10)

,supplier_id,avg_actual_lead_time,avg_expected_lead_time,avg_lead_time_variance,lead_time_std
32,SUP033,97.37,90.90,6.47,10.09
38,SUP039,44.05,41.70,2.35,8.89
1,SUP002,64.90,62.56,2.34,10.91
20,SUP021,61.91,59.63,2.28,6.95
22,SUP023,57.77,55.51,2.26,9.47
23,SUP024,58.53,56.27,2.26,9.39
5,SUP006,60.85,58.71,2.15,7.20
24,SUP025,46.58,44.47,2.11,6.82
8,SUP009,52.03,49.92,2.11,10.17
4,SUP005,53.51,51.41,2.10,7.43
